In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import os
import torch
from amr.utils import logger

# 设置matplotlib后端为Agg（非交互式，适合服务器环境）
matplotlib.use('Agg')

__all__ = ["draw_train", "draw_conf", "draw_acc"]


def draw_train(train_loss, train_acc, valid_loss, valid_acc, save_path):
    """
    绘制训练和验证的损失与准确率曲线

    train_loss: 训练损失列表（每个元素对应一个epoch）
    train_acc: 训练准确率列表
    valid_loss: 验证损失列表
    valid_acc: 验证准确率列表
    save_path: 图片保存路径
    """

    # 创建12英寸宽、4英寸高的画布
    plt.figure(figsize=(12, 4))

    # 将画布分为 1 行 2 列，当前操作左图
    plt.subplot(1, 2, 1)

    # 绘制损失曲线
    # "ro-"：红色圆点连线
    plt.plot(train_loss, "ro-", label="Train loss")
    # "bs-"：蓝色方块连线
    plt.plot(valid_loss, "bs-", label="Val loss")

    # 显示图例
    plt.legend()

    # x轴标签
    plt.xlabel("epoch")
    # y轴标签
    plt.ylabel("loss")

    # 切换到第右图
    plt.subplot(1, 2, 2)

    # 绘制准确率曲线
    plt.plot(train_acc, "ro-", label="Train acc")
    plt.plot(valid_acc, "bs-", label="Val acc")

    plt.xlabel("epoch")
    plt.ylabel("acc")

    plt.legend()

    # 创建保存路径（若不存在）
    os.makedirs(save_path, exist_ok=True)

    # 保存图片
    plt.savefig(os.path.join(save_path, 'train_process.jpg'))

    # 记录日志
    logger.info(f'save the draw of training.')

    # 补充
    # 添加网格线
    plt.grid(True)
    # 支持中文
    plt.rcParams["font.family"] = ["SimHei", "WenQuanYi Micro Hei", "Heiti TC"]

    # 关闭图表，释放内存
    plt.close()


def draw_conf(test_conf, save_path, cmap=plt.cm.Blues, labels=[], order=None):
    '''
    绘制混淆矩阵

    test_conf: 混淆矩阵数据（torch.Tensor或numpy.ndarray）
    save_path: 图片保存路径
    cmap: 颜色映射（默认蓝色系）
    labels: 类别标签列表（如["猫", "狗", "鸟"]）
    order: 序号（用于区分不同混淆矩阵，如"final"）
    '''


    plt.figure(figsize=(16, 16))

    # 使用imshow显示矩阵数据
    # interpolation='nearest'：不使用插值，每个单元格颜色均匀。
    # cmap=plt.cm.Blues：使用蓝色系颜色映射（数值越大颜色越深）。
    plt.imshow(test_conf, interpolation='nearest', cmap=cmap)

    plt.title("Confusion matrix")

    plt.colorbar()

    # 生成类别索引（如[0, 1, 2]）
    tick_marks = torch.arange(len(labels))
    # x轴标签（预测类别）
    plt.xticks(tick_marks, labels, rotation=45, fontsize=10)
    # y轴标签（真实类别）
    plt.yticks(tick_marks, labels, fontsize=10)

    # 将Tensor转为numpy数组并保留两位小数
    test_conf = test_conf.numpy().round(2)

    for i in range(len(labels)):
        for j in range(len(labels)):
            if test_conf[i,j] == 0:
                continue
            if i == j:
                # 对角线元素（正确分类）
                # va/ha='center' - 文本垂直/水平居中
                # color='white' - 白色文字（深蓝色背景上更醒目）
                plt.text(j, i, test_conf[i, j], va='center', ha='center', color='white')
            else:
                # 非对角线元素（错误分类）
                # 默认黑色文字
                plt.text(j, i, test_conf[i, j], va='center', ha='center')

    # 自动调整子图参数，避免标签被截断
    plt.tight_layout()

    plt.ylabel('True label')

    plt.xlabel('Predicted label')

    # 创建保存目录，exist_ok=True表示目录已存在时不报错
    os.makedirs(save_path, exist_ok=True)

    # 保存图片，文件名格式：test_conf_[order].jpg
    plt.savefig(os.path.join(save_path, 'test_conf_'+order+'.jpg'))

    # 记录日志信息（假设logger已定义）
    logger.info(f'save the draw of confu sion matrix.')

    # 关闭图形，释放内存
    plt.close()


def draw_acc(snrs, test_acc_snr, save_path):
    '''
    绘制测试的信噪比(SNR)与测试准确率(Accuracy)的关系曲线

    snrs: 信噪比数组(dB)，如[-20, -10, 0, 10, 20]
    test_acc_snr: 对应各SNR的测试准确率数组，如[0.2, 0.5, 0.8, 0.9, 0.95]
    save_path: 图片保存路径
    '''
    plt.figure(figsize=(6, 4))

    plt.plot(snrs, test_acc_snr, "bs-", label="Test acc")
    plt.xlabel("snr(dB)")
    plt.ylabel("acc")
    plt.legend()
    # 创建保存目录，exist_ok=True表示目录已存在时不报错
    os.makedirs(save_path, exist_ok=True)
    # 保存图片，文件名格式：test_conf_[order].jpg
    plt.savefig(os.path.join(save_path, 'test_acc.jpg'))
    # 记录日志信息（假设logger已定义）
    logger.info(f'save the draw of testing accuracy.')
    # 关闭图形，释放内存
    plt.close()